# Hybrid RRNCO-PARCO for the HCVRP

Learning a Parallel AutoRegressive policy with an Attention-Free encoder for the Heterogeneous Capacitated Vehicle Routing Problem (HCVRP) with real-world distance matrices.

The **Hybrid** model combines:
- **RRNCO**'s attention-free encoder (with Neural Adaptive Bias for distance matrices)
- **PARCO**'s multi-agent parallel autoregressive decoder

This notebook mirrors the structure of the [PARCO quickstart](../parco/examples/1.quickstart-hcvrp.ipynb) but uses the hybrid architecture.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
from rl4co.utils.trainer import RL4COTrainer

from hybrid.envs import HCVRPRealEnv
from hybrid.models import HybridRLModule, HybridPolicy

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Environment

`HCVRPRealEnv` extends PARCO's `HCVRPEnv` with distance matrix support. It can use:
- **Euclidean distances** (auto-computed from coordinates — used here for demo)
- **Real-world OSRM distances** (from pre-computed city data)

In [ ]:
# Create the hybrid environment (Euclidean mode — no city data needed for demo)
env = HCVRPRealEnv(
    generator_params=dict(num_loc=60, num_agents=5),
)

# Generate a small batch of test instances
td_test_data = env.generator(batch_size=[3])
td_init = env.reset(td_test_data.clone()).to(device)
td_init_test = td_init.clone()

print(f"Test data keys: {list(td_init.keys())}")
print(f"Locations shape: {td_init['locs'].shape}")
print(f"Distance matrix shape: {td_init['distance_matrix'].shape}")

## Model

Here we declare our hybrid policy and RL module (policy + environment + RL algorithm).

- `HybridPolicy`: RRNCO's attention-free encoder + PARCO's multi-agent decoder
- `HybridRLModule`: Full training wrapper with SymNCO losses, augmentation, and curriculum sampling

In [ ]:
emb_dim = 128

# Hybrid Policy: RRNCO encoder + PARCO decoder
policy = HybridPolicy(
    env_name="hcvrp_real",
    embed_dim=emb_dim,
    num_encoder_layers=6,
    num_heads=8,
    agent_handler="highprob",
    use_distance_bias=True,
    fusion_type="concat",
    normalization_encoder="instance",
    context_embedding_kwargs={
        "normalization": "rms",
        "norm_after": False,
    },
)

# Full RL module: policy + environment + training config
model = HybridRLModule(
    env,
    policy=policy,
    train_data_size=10_000,       # Small size for demo
    val_data_size=1000,           # Small size for demo
    batch_size=64,                # Small size for demo
    num_augment=4,                # SymNCO augments
    train_min_agents=5,
    train_max_agents=5,
    train_min_size=60,
    train_max_size=60,
    optimizer_kwargs={'lr': 1e-4, 'weight_decay': 0},
)

### Test untrained model

Before training, we evaluate the randomly-initialized model and visualize the (random) routes.

In [ ]:
policy = model.policy.to(device)
out = policy(td_init_test.clone(), env, decode_type="greedy", return_actions=True)

# Plotting
actions = out["actions"]
print("Average tour length (untrained): {:.2f}".format(-out['reward'].mean().item()))
for i in range(3):
    print(f"Tour {i} length: {-out['reward'][i].item():.2f}")
    env.render(td_init[i], actions[i].cpu())

## Training

In here we call the trainer and then fit the model.

In [ ]:
trainer = RL4COTrainer(
    max_epochs=5,  # few epochs for demo
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    logger=None,
)

In [ ]:
trainer.fit(model)

## Evaluating the trained model

Now, we evaluate the trained model with different decoding strategies:
- **Greedy**: Take the action with the highest probability
- **Sampling**: Sample from the probability distribution N times and take the best one

### Greedy evaluation

Here we simply take the solution with greedy decoding type.

In [ ]:
# Use same initial td as before
model = model.to(device)
out_greedy = model(td_init_test.clone(), phase="test", decode_type="greedy", return_actions=True)

# Plotting
actions_greedy = out_greedy["actions"]
print("Average tour length (greedy): {:.2f}".format(-out_greedy['reward'].mean().item()))
for i in range(3):
    print(f"Tour {i} length: {-out_greedy['reward'][i].item():.2f}")
    env.render(td_init[i], actions_greedy[i].cpu())

### Sampling evaluation

We sample multiple solutions and keep the best one.

In [ ]:
num_samples = 10

with torch.no_grad():
    out_sampling = model.policy(
        td_init_test.clone(), env,
        decode_type="sampling", num_samples=num_samples,
        return_actions=True,
    )

# Reshape rewards: [batch * num_samples] -> [batch, num_samples]
rewards = out_sampling['reward'].reshape(3, num_samples)
best_idx = rewards.argmax(dim=-1)  # best sample per instance

# Get best actions for each instance
actions_sampling = out_sampling['actions'].reshape(
    3, num_samples, *out_sampling['actions'].shape[1:]
)
best_rewards = rewards.gather(1, best_idx.unsqueeze(-1)).squeeze(-1)

print("Average tour length (sampling, best of {}): {:.2f}".format(
    num_samples, -best_rewards.mean().item()))

for i in range(3):
    best_action = actions_sampling[i, best_idx[i]]
    print(f"Tour {i} length: {-best_rewards[i].item():.2f}")
    env.render(td_init[i], best_action.cpu())

## Side-by-side Comparison

Compare the untrained vs. trained (greedy vs. sampling) routes for a single instance.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import cm


def render_on_ax(ax, td, actions, title=None):
    """Render HCVRP solution on a given matplotlib Axes."""
    td = td.detach().cpu()
    if td.batch_size != torch.Size([]):
        td = td[0]
        actions = actions[0]

    num_agents = td["current_node"].size(-1)

    # Depot
    ax.scatter(td["locs"][0, 0], td["locs"][0, 1],
               marker="s", color="r", s=100, zorder=5, label="Depot")

    # Customers
    ax.scatter(td["locs"][num_agents:, 0], td["locs"][num_agents:, 1],
               marker="o", color="tab:gray", s=25, label="Customers")

    # Routes per agent
    depot_first = torch.arange(num_agents).unsqueeze(0).expand(actions.size(0), -1)
    actions_full = torch.cat([depot_first, actions, depot_first], dim=-1)

    for agent_idx in range(num_agents):
        agent_action = actions_full[agent_idx]
        for j in range(agent_action.size(0) - 1):
            src = td["locs"][agent_action[j]]
            dst = td["locs"][agent_action[j + 1]]
            is_depot = (agent_action[j] == agent_idx) or (agent_action[j + 1] == agent_idx)
            if is_depot:
                ax.plot([src[0], dst[0]], [src[1], dst[1]],
                        color=cm.Set2(agent_idx), lw=0.4, ls="--")
            else:
                ax.plot([src[0], dst[0]], [src[1], dst[1]],
                        color=cm.Set2(agent_idx), lw=1.2)

    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xticks([])
    ax.set_yticks([])
    if title:
        ax.set_title(title, fontsize=11)


# ── Plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

render_on_ax(
    axes[0], td_init[0], out['actions'][0].cpu(),
    title=f"Untrained\nCost: {-out['reward'][0].item():.2f}",
)
render_on_ax(
    axes[1], td_init[0], out_greedy['actions'][0].cpu(),
    title=f"Trained (Greedy)\nCost: {-out_greedy['reward'][0].item():.2f}",
)
best_action_0 = actions_sampling[0, best_idx[0]]
render_on_ax(
    axes[2], td_init[0], best_action_0.cpu(),
    title=f"Trained (Sampling ×{num_samples})\nCost: {-best_rewards[0].item():.2f}",
)

plt.suptitle("Hybrid RRNCO-PARCO: Untrained vs Trained", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Loading from a Checkpoint

If you have a trained checkpoint (`.ckpt` file), you can load it and run inference + visualization directly.
Update the `checkpoint_path` below and uncomment the cell.

In [ ]:
# ── Uncomment and update the path to your checkpoint ─────────────────
# checkpoint_path = "logs/train/runs/<your-run>/checkpoints/last.ckpt"
#
# model_loaded = HybridRLModule.load_from_checkpoint(
#     checkpoint_path, map_location="cpu", strict=False
# )
# policy_loaded = model_loaded.policy.to(device).eval()
#
# # Load test data from .npz file
# from rl4co.data.utils import load_npz_to_tensordict
# td_test = load_npz_to_tensordict("data/hcvrp_real/test_real_n100_m7.npz")
#
# # Pick first 3 instances for visualization
# td_small = td_test[:3]
# env_eval = HCVRPRealEnv()
# td_reset = env_eval.reset(td_small).to(device)
#
# with torch.no_grad():
#     out_ckpt = policy_loaded(
#         td_reset.clone(), env_eval,
#         decode_type="greedy", return_actions=True
#     )
#
# actions_ckpt = out_ckpt["actions"]
# print("Average tour length: {:.2f}".format(-out_ckpt['reward'].mean().item()))
# for i in range(3):
#     print(f"Tour {i} length: {-out_ckpt['reward'][i].item():.2f}")
#     env_eval.render(td_reset[i], actions_ckpt[i].cpu())

print("Uncomment this cell and set checkpoint_path to load a trained model.")